In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/aes-enem-replication
!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1wMnxscg1-8nkk6NpKvn7kHGGzrM-lvgp/aes-enem-replication


In [2]:
!PYTHONPATH=. python -m src.data.preprocessing --dataset essay_br

Processando EssayBR a partir de data/processed/essay_br/clean_essay_br.parquet ...
  train: 5261 redações
  val: 658 redações
  test: 658 redações
EssayBR processado com sucesso!


In [5]:
import glob, os, shutil

# C1..C5 = versões 24..28 (última execução, lr=3e-5)
for i, v in enumerate(range(24, 29), start=1):
    src = glob.glob(f"lightning_logs/version_{v}/checkpoints/*.ckpt")
    assert len(src) == 1, f"version_{v}: esperava 1 checkpoint, achei {src}"
    dst = f"checkpoints/aes_enem/c{i}/best.ckpt"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(src[0], dst)
    print(f"c{i} <- version_{v} ({os.path.getsize(dst)/1e9:.2f} GB)")

c1 <- version_24 (1.30 GB)
c2 <- version_25 (1.30 GB)
c3 <- version_26 (1.30 GB)
c4 <- version_27 (1.30 GB)
c5 <- version_28 (1.30 GB)


In [6]:
!PYTHONPATH=. python -m src.train --skip-training

Treino: aes_enem | Teste: aes_enem (test) | Competências: c1, c2, c3, c4, c5
Loading weights: 100% 199/199 [00:00<00:00, 14511.40it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
--- Resultados para Competência C1 ---
QWK:

In [7]:
!PYTHONPATH=. python -m src.train --train-dataset aes_enem --test-dataset essay_br --test-split all --skip-training

Treino: aes_enem | Teste: essay_br (all) | Competências: c1, c2, c3, c4, c5
Loading weights: 100% 199/199 [00:00<00:00, 20949.94it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
--- Resultados para Competência C1 ---
QWK: 

resultado limpo (só redações novas)

In [8]:
!PYTHONPATH=. python -m src.data.preprocessing --dataset essay_br
!PYTHONPATH=. python -m src.train --train-dataset aes_enem --test-dataset essay_br --test-split all --skip-training --exclude-overlap

Processando EssayBR a partir de data/processed/essay_br/clean_essay_br.parquet ...
  383 de 6577 redações do EssayBR também estão no AES-ENEM
  train: 5261 redações
  val: 658 redações
  test: 658 redações
EssayBR processado com sucesso!
Treino: aes_enem | Teste: essay_br (all) | Competências: c1, c2, c3, c4, c5
Loading weights: 100% 199/199 [00:00<00:00, 19719.95it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPE